# cluster: Multi-Notebook Distributed GPU Mesh (Leader Orchestrator)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/berlin-566/cluster/blob/main/notebooks/colab_cluster.ipynb)

Pool multiple ephemeral cloud notebooks (Google Colab free T4, Kaggle 2x T4, team GPUs) into a single unified multi-GPU inference cluster.

### How It Works:
1. **Primary Leader (This Notebook)**: Manages Cloudflare Tunnel, coordinates multi-GPU RPC mesh, distributes layer shards (llama.cpp RPC), and serves the control board.
2. **Secondary Worker Notebooks**: Open blank Colab or Kaggle notebooks, paste the generated one-line command to attach their GPUs.
3. **Local Machine**: Manage via `cluster admin connect <url>` or launch the sleek Cortex.io-style matte-black web dashboard with `cluster ui`.

### Instructions:
1. **Switch to GPU**: Click **Runtime** -> **Change runtime type** -> Select **T4 GPU**.
2. Set the number of worker notebooks below (default: 3 workers = 4 GPUs total, 64GB pooled VRAM).
3. Click **Run All** (or `Ctrl+F9`).
4. Copy the Worker Join commands into your blank worker notebooks.

In [ ]:
# @title Leader Mesh Coordinator { display-mode: "form" }
WORKER_NOTEBOOKS_COUNT = 3 # @param {type:"integer"}
PERSIST_TO_GOOGLE_DRIVE = False # @param {type:"boolean"}

import os
import subprocess

os.environ['CLUSTER_EXPECTED_WORKERS'] = str(WORKER_NOTEBOOKS_COUNT)

if PERSIST_TO_GOOGLE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    models_dir = '/content/drive/MyDrive/cluster_models'
    os.makedirs(models_dir, exist_ok=True)
    os.environ['OLLAMA_MODELS'] = models_dir
    print(f"[STORAGE] Persistent model storage enabled at: {models_dir}")

# Keepalive in background
from IPython.display import display, Javascript, HTML
display(HTML('''
<script>
function keepAlive() {
   console.log("[cluster] Mesh heartbeat active");
}
setInterval(keepAlive, 60000);
</script>
'''))

# Fetch and run server supervisor
!curl -fsSL https://raw.githubusercontent.com/berlin-566/cluster/main/remote/bootstrap.sh | bash


### Worker Notebook Template

To connect a secondary blank Colab/Kaggle notebook as a worker accelerator node, run:
```bash
!curl -fsSL https://raw.githubusercontent.com/berlin-566/cluster/main/remote/worker.sh | bash -s -- --leader <LEADER_URL> --token <WORKER_TOKEN> --name notebook-X
```